# Day 09. Exercise 00
# Regularization

## 0. Imports

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from joblib import dump
from joblib import load
import time


## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [4]:
df = pd.read_csv("../data/dayofweek.csv")
df

,numTrials,hour,dayofweek,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,-0.788667,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.756764,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.724861,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.692958,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.661055,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,-0.533442,0.945382,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1682,-0.629151,0.945382,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1683,-0.597248,0.945382,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1684,-0.565345,0.945382,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [5]:
X = df.drop('dayofweek', axis=1)
y = df["dayofweek"]
X, X_test, y, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [2]:
def stratified_k_fold(n_splits):
    skf = StratifiedKFold(n_splits=n_splits)
    model = LogisticRegression(random_state=21, fit_intercept=False)

    accuracies_valid = []
    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model.fit(X_train, y_train)

        accuracy_train = model.score(X_train, y_train)
        accuracy_valid = model.score(X_test, y_test)
        
        print(f"train - {accuracy_train:.5f} | valid - {accuracy_valid:.5f}")
        accuracies_valid.append(accuracy_valid)

    print(f"Average accuracy on crossval is {np.mean(accuracies_valid):.5f}")
    print(f"Std is {np.std(accuracies_valid):.5f}")



In [6]:
%%time
stratified_k_fold(n_splits=10) 


train - 0.62902 | valid - 0.59259
train - 0.64633 | valid - 0.62963
train - 0.63479 | valid - 0.56296
train - 0.65622 | valid - 0.61481
train - 0.63397 | valid - 0.57778
train - 0.64056 | valid - 0.59259
train - 0.64138 | valid - 0.65926
train - 0.65952 | valid - 0.56296
train - 0.64333 | valid - 0.59701
train - 0.63674 | valid - 0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
CPU times: user 1.93 s, sys: 2.97 s, total: 4.9 s
Wall time: 1.27 s


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [8]:
%%time
model = LogisticRegression(fit_intercept=False, random_state=21, penalty='l1', solver='saga')
model.fit(X, y)

accuracy_score = model.score(X_test, y_test)
accuracy_score

CPU times: user 297 ms, sys: 5.32 ms, total: 303 ms
Wall time: 307 ms


/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn("The max_iter was reached which means "


0.6331360946745562

In [9]:
%%time
model = LogisticRegression(fit_intercept=False, random_state=21, penalty='l2')
model.fit(X, y)

accuracy_score = model.score(X_test, y_test)
accuracy_score

CPU times: user 143 ms, sys: 234 ms, total: 378 ms
Wall time: 74.2 ms


0.6272189349112426

In [10]:
%%time
model = LogisticRegression(fit_intercept=False, random_state=21, penalty='none', solver='newton-cg')
model.fit(X, y)

accuracy_score = model.score(X_test, y_test)
accuracy_score

CPU times: user 1.71 s, sys: 2.39 s, total: 4.1 s
Wall time: 719 ms


0.650887573964497

## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [12]:
%%time
def stratified_k_fold_svm(n_splits):
    model = SVC(probability=True, kernel='linear', random_state=21)
    skf = StratifiedKFold(n_splits=n_splits)

    accuracy_score = []
    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        
        model.fit(X_train, y_train)
        acuuracy_train = model.score(X_train, y_train)
        accuracy_test = model.score(X_test, y_test)

        print(f"train - {acuuracy_train:.5f} | valid - {accuracy_test:.5f}")
        accuracy_score.append(accuracy_test)

    print(f"Average accuracy on crossval is {np.mean(accuracy_score):.5f}")
    print(f"Std is {np.std(accuracy_score):.5f}")

stratified_k_fold_svm(n_splits=10)

train - 0.70486 | valid - 0.65926
train - 0.69662 | valid - 0.75556
train - 0.69415 | valid - 0.62222
train - 0.70239 | valid - 0.65185
train - 0.69085 | valid - 0.65185
train - 0.68920 | valid - 0.64444
train - 0.69250 | valid - 0.72593
train - 0.70074 | valid - 0.62222
train - 0.69605 | valid - 0.61940
train - 0.71087 | valid - 0.63433
Average accuracy on crossval is 0.65871
Std is 0.04359
CPU times: user 3.92 s, sys: 36 ms, total: 3.96 s
Wall time: 3.97 s


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [13]:
%%time
model = SVC(probability=True, kernel='linear', random_state=21, C=1.0)
model.fit(X, y)

accuracy = model.score(X_test, y_test)
accuracy

CPU times: user 495 ms, sys: 10.4 ms, total: 505 ms
Wall time: 527 ms


0.7159763313609467

In [14]:
%%time
model = SVC(probability=True, kernel='linear', random_state=21, C=0.1)
model.fit(X, y)

accuracy = model.score(X_test, y_test)
accuracy

CPU times: user 444 ms, sys: 7.37 ms, total: 452 ms
Wall time: 460 ms


0.5976331360946746

In [15]:
%%time
model = SVC(probability=True, kernel='linear', random_state=21, C=100)
model.fit(X, y)

accuracy = model.score(X_test, y_test)
accuracy

CPU times: user 2.39 s, sys: 25.1 ms, total: 2.41 s
Wall time: 2.56 s


0.7662721893491125

## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [16]:
%%time
def stratified_k_fold_svm(n_splits):
    model = DecisionTreeClassifier(max_depth=10, random_state=21)
    skf = StratifiedKFold(n_splits=n_splits)

    accuracy_score = []
    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        
        model.fit(X_train, y_train)
        acuuracy_train = model.score(X_train, y_train)
        accuracy_test = model.score(X_test, y_test)

        print(f"train - {acuuracy_train:.5f} | valid - {accuracy_test:.5f}")
        accuracy_score.append(accuracy_test)

    print(f"Average accuracy on crossval is {np.mean(accuracy_score):.5f}")
    print(f"Std is {np.std(accuracy_score):.5f}")

stratified_k_fold_svm(n_splits=10)

train - 0.81039 | valid - 0.74074
train - 0.77741 | valid - 0.74074
train - 0.83347 | valid - 0.70370
train - 0.79720 | valid - 0.76296
train - 0.82440 | valid - 0.75556
train - 0.80379 | valid - 0.68889
train - 0.80709 | valid - 0.76296
train - 0.80132 | valid - 0.65926
train - 0.80807 | valid - 0.75373
train - 0.80478 | valid - 0.68657
Average accuracy on crossval is 0.72551
Std is 0.03562
CPU times: user 86.5 ms, sys: 4.79 ms, total: 91.3 ms
Wall time: 93.5 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [17]:
%%time
model = DecisionTreeClassifier(max_depth=12, random_state=21, min_samples_split=10, min_samples_leaf=5)
model.fit(X, y)

accuracy = model.score(X_test, y_test)
accuracy

CPU times: user 9.97 ms, sys: 2.78 ms, total: 12.8 ms
Wall time: 12.5 ms


0.7248520710059172

In [18]:
%%time
model = DecisionTreeClassifier(max_depth=15, random_state=21, min_samples_split=12, min_samples_leaf=4)
model.fit(X, y)

accuracy = model.score(X_test, y_test)
accuracy

CPU times: user 8.61 ms, sys: 1.88 ms, total: 10.5 ms
Wall time: 9.21 ms


0.7810650887573964

In [19]:
%%time
model = DecisionTreeClassifier(max_depth=20, random_state=21, min_samples_split=12, min_samples_leaf=1)
model.fit(X, y)

accuracy = model.score(X_test, y_test)
accuracy

CPU times: user 10.9 ms, sys: 2.88 ms, total: 13.8 ms
Wall time: 13.3 ms


0.8402366863905325

## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [20]:
%%time
def stratified_k_fold_svm(n_splits):
    model = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
    skf = StratifiedKFold(n_splits=n_splits)

    accuracy_score = []
    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        
        model.fit(X_train, y_train)
        acuuracy_train = model.score(X_train, y_train)
        accuracy_test = model.score(X_test, y_test)

        print(f"train - {acuuracy_train:.5f} | valid - {accuracy_test:.5f}")
        accuracy_score.append(accuracy_test)

    print(f"Average accuracy on crossval is {np.mean(accuracy_score):.5f}")
    print(f"Std is {np.std(accuracy_score):.5f}")

stratified_k_fold_svm(n_splits=10)

train - 0.96455 | valid - 0.88148
train - 0.96208 | valid - 0.91852
train - 0.96785 | valid - 0.86667
train - 0.96455 | valid - 0.89630
train - 0.96538 | valid - 0.91111
train - 0.96538 | valid - 0.88148
train - 0.97115 | valid - 0.91852
train - 0.96867 | valid - 0.85185
train - 0.97364 | valid - 0.88060
train - 0.97941 | valid - 0.86567
Average accuracy on crossval is 0.88722
Std is 0.02204
CPU times: user 1.19 s, sys: 26.2 ms, total: 1.22 s
Wall time: 1.24 s


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [21]:
%%time
model = RandomForestClassifier(n_estimators=100, max_depth=14, random_state=21)
model.fit(X, y)

accuracy = model.score(X_test, y_test)
accuracy

CPU times: user 213 ms, sys: 7.26 ms, total: 220 ms
Wall time: 224 ms


0.8964497041420119

In [22]:
%%time
model = RandomForestClassifier(n_estimators=150, max_depth=14, random_state=21, min_samples_split=3, min_samples_leaf=1, criterion='entropy')
model.fit(X, y)

accuracy = model.score(X_test, y_test)
accuracy

CPU times: user 342 ms, sys: 7.88 ms, total: 350 ms
Wall time: 359 ms


0.9142011834319527

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [23]:
%%time
model = RandomForestClassifier(n_estimators=150, max_depth=14, random_state=21, min_samples_split=3, min_samples_leaf=1, criterion='entropy')
model.fit(X, y)

y_pred = pd.Series(model.predict(X_test))
accuracy = model.score(X_test, y_test)
accuracy

CPU times: user 374 ms, sys: 11.1 ms, total: 385 ms
Wall time: 396 ms


0.9142011834319527

In [ ]:
%%time
error = [0]* 7
for i in range(y_pred.shape[0]):
    if y_test.iloc[i] != y_pred.iloc[i]:
        error[y_test.iloc[i]] += 1

max_error = max(error)
day_with_max_errors = [i for i in range(len(error)) if error[i] == max_error]



CPU times: user 7.34 ms, sys: 1.13 ms, total: 8.47 ms
Wall time: 9.43 ms


[7, 6, 2, 2, 3, 5, 4]

In [25]:
%%time
dump(model, "../data/best_model.joblib", compress=9)

CPU times: user 2.58 s, sys: 17.6 ms, total: 2.6 s
Wall time: 2.61 s


['../data/best_model.joblib']